## Read annotation JSONL files

This cell loads the `.jsonl` files created by `annotation.py`, reads them in sorted order, and prints each line.

In [1]:
from pathlib import Path
import json
import os
from tqdm import tqdm

jsonl_dir = Path("/data/user_data/yixiz/autonlab/zfsauton/scratch/mineuih/waymax_rs/annotations/target_lane/")
target_dir = Path("/data/user_data/yixiz/autonlab/zfsauton/scratch/mineuih/waymax_rs/annotations_with_functions")
target_dir.mkdir(parents=True, exist_ok=True)
jsonl_files = sorted(jsonl_dir.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files in {jsonl_dir}")
for jsonl_path in jsonl_files:
    if "00000" in jsonl_path.name:
        print(f"Found JSONL file: {jsonl_path.name}")

if not jsonl_files:
    print(f"No JSONL files found in {jsonl_dir}")
else:
    for jsonl_path in tqdm(jsonl_files, desc="Processing JSONL files"):
        output_jsonl_path = target_dir / jsonl_path.name
        output_idx_path = target_dir / jsonl_path.with_suffix(".idx.json").name
        temp_jsonl_path = output_jsonl_path.with_suffix(output_jsonl_path.suffix + ".tmp")
        temp_idx_path = output_idx_path.with_suffix(output_idx_path.suffix + ".tmp")
        offsets = []

        with jsonl_path.open("r", encoding="utf-8") as reader, temp_jsonl_path.open("w", encoding="utf-8") as writer:
            offset = 0
            for line_number, line in enumerate(reader, start=1):
                line = line.rstrip("\n")
                if not line:
                    continue
                data = json.loads(line)
                annot = data["annotation"]["ego_motion"]
                functions = []
                if float(annot['start_speed']) < 0.1:
                    if float(annot['end_speed']) < 0.1:
                        functions.append('stop')
                    elif float(annot['end_speed']) > 0.1:
                        functions.append('stop_and_go')
                elif float(annot['start_speed']) < float(annot['end_speed']):
                    functions.append('accelerate')
                else:
                    functions.append('decelerate')

                if 'left' in annot['turn_label']:
                    functions.append('turn_left')
                elif 'right' in annot['turn_label']:
                    functions.append('turn_right')

                if 'left' in annot['lane_change']:
                    functions.append('change_lane_left')
                elif 'right' in annot['lane_change']:
                    functions.append('change_lane_right')
                elif 'none' in annot['lane_change']:
                    functions.append('follow_current_lane')

                functions_text = ', '.join(functions)
                data['functions'] = functions_text

                line_text = json.dumps(data, ensure_ascii=False)
                offsets.append(offset)
                writer.write(line_text)
                writer.write("\n")
                offset += len(line_text.encode("utf-8")) + 1
                # print(f"[{line_number}] {functions_text}")

        with temp_idx_path.open("w", encoding="utf-8") as idx_writer:
            json.dump(offsets, idx_writer)
            idx_writer.write("\n")

        os.replace(temp_jsonl_path, output_jsonl_path)
        os.replace(temp_idx_path, output_idx_path)


Found 6000 JSONL files in /data/user_data/yixiz/autonlab/zfsauton/scratch/mineuih/waymax_rs/annotations/target_lane
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t0.jsonl
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t10.jsonl
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t20.jsonl
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t30.jsonl
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t40.jsonl
Found JSONL file: training_tfexample.tfrecord-00000-of-01000_t50.jsonl


Processing JSONL files: 100%|██████████| 6000/6000 [05:38<00:00, 17.70it/s]
